# 05_llmip_financial.ipynb

## LLMIP Pipeline for Financial (Stateful) Experiment

This is **Experiment 2** from the thesis - the **STATEFUL** environment.

Key difference from grid:
- Target has **memory** (path dependency, autocorrelation)
- Predicting next-day return direction (UP/DOWN)
- This stress-tests whether LLMIP works in stateful environments

API: OpenRouter (z-ai/glm-4.7)

## 1. Setup

In [1]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
load_dotenv("/home/l1zle/LLMIP/.env")
import requests
import time

PROJECT_DIR = Path("/home/l1zle/LLMIP")
FIN_DIR = PROJECT_DIR / "data" / "financial"
RESULTS_DIR = PROJECT_DIR / "results"

# OpenRouter API
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
MODEL = "z-ai/glm-4.7"
BASE_URL = "https://openrouter.ai/api/v1"

print(f"API Key: {'Set' if OPENROUTER_API_KEY else 'NOT SET'}")
print(f"Model: {MODEL}")

API Key: Set
Model: z-ai/glm-4.7


## 2. Load Data

In [2]:
train = pd.read_csv(FIN_DIR / "sp500_train.csv")
test = pd.read_csv(FIN_DIR / "sp500_test.csv")

train_targets = train['target_direction']
test_targets = test['target_direction']

exclude = ['target_direction', 'next_return', 'date']
feature_cols = [c for c in train.columns if c not in exclude]
train_features = train[feature_cols]
test_features = test[feature_cols]

print(f"Train: {train_features.shape}")
print(f"Test:  {test_features.shape}")
print(f"\nTarget: 0=DOWN, 1=UP")
print(f"Train distribution: {train_targets.value_counts().to_dict()}")

Train: (764, 23)
Test:  (192, 23)

Target: 0=DOWN, 1=UP
Train distribution: {1: 401, 0: 363}


## 3. LLM Client

In [3]:
def call_llm(prompt, system=None, max_tokens=500, temperature=0.3):
    """Call OpenRouter API."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": "https://llmip.research",
        "X-Title": "LLMIP Research"
    }
    
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    payload = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature
    }
    
    try:
        resp = requests.post(f"{BASE_URL}/chat/completions", headers=headers, json=payload, timeout=120)
        if resp.status_code == 200:
            result = resp.json()
            msg = result["choices"][0]["message"]
            # Try content first, then reasoning
            return msg.get("content") or msg.get("reasoning") or ""
        else:
            print(f"Error: {resp.status_code} - {resp.text[:100]}")
    except Exception as e:
        print(f"Exception: {e}")
    return None

# Test
test_resp = call_llm("Say 'Test OK' in exactly those words.")
print(f"Test response: {test_resp[:50] if test_resp else 'None'}")

Test response: TestOK


## 4. Phase 1: Training Analysis

In [4]:
print("\n" + "="*60)
print("PHASE 1: TRAINING ANALYSIS")
print("="*60)

n_samples = 20
subset = train_features.iloc[:n_samples]
target_subset = train_targets.iloc[:n_samples]

feature_desc = """
- return_1d: 1-day return (lagged)
- return_5d: 5-day return
- return_20d: 20-day return
- volatility_5d: 5-day volatility
- volatility_20d: 20-day volatility
- rsi: Relative Strength Index (0-100)
- momentum: Momentum indicator
- macd: MACD line
- bb_position: Bollinger Band position (0-1)"""

prompt = f"""Analyze this market data to identify patterns predicting next-day return direction.

Features:
{feature_desc}

Training samples:
{subset[['return_1d', 'return_5d', 'volatility_5d', 'rsi', 'bb_position']].to_string()}

Targets (0=DOWN, 1=UP): {target_subset.tolist()}

Identify 3 key patterns that predict direction. Keep under 100 words."""

print(f"Analyzing {n_samples} samples...")
analysis = call_llm(prompt, max_tokens=300)

if analysis:
    with open(RESULTS_DIR / "financial_phase1_analysis.txt", "w") as f:
        f.write(analysis)
    print(f"\nPhase 1 complete! ({len(analysis)} chars)")
    print(analysis[:500])
else:
    print("Phase 1 failed!")


PHASE 1: TRAINING ANALYSIS
Analyzing 20 samples...

Phase 1 complete! (933 chars)
1.  **Analyze the Request:**
    *   **Goal:** Identify 3 key patterns predicting next-day return direction (UP/DOWN) from the provided market data.
    *   **Input:** A dataset with 20 samples and 6 features (return_1d, return_5d, volatility_5d, rsi, bb_position). Note: The prompt lists return_20d, volatility_20d, momentum, macd in the feature list but the provided data *only* contains return_1d, return_5d, volatility_5d, rsi, bb_position. I must work with the provided data columns.
    *   **O


## 5. Phase 2: Predictions

In [5]:
print("\n" + "="*60)
print("PHASE 2: PREDICTIONS")
print("="*60)

n_test = 5
predictions = {}

for i in range(n_test):
    row = test_features.iloc[i]
    
    prompt = f"""Predict next-day return direction for this market data.
    
Features:
- return_1d: {row['return_1d']:.4f}
- return_5d: {row['return_5d']:.4f}
- volatility_5d: {row['volatility_5d']:.4f}
- rsi: {row['rsi']:.2f}
- bb_position: {row['bb_position']:.4f}
    
Predict: 0 (DOWN/negative return) or 1 (UP/positive return)
Just output the number 0 or 1, nothing else."""
    
    resp = call_llm(prompt, max_tokens=10, temperature=0.0)
    
    # Extract 0 or 1
    pred = None
    if resp:
        if '1' in resp: pred = 1
        elif '0' in resp: pred = 0
    
    predictions[str(i)] = {
        "raw": resp or "",
        "prediction": pred,
        "actual": int(test_targets.iloc[i])
    }
    
    correct = pred == int(test_targets.iloc[i]) if pred is not None else False
    print(f"  Sample {i}: pred={pred}, actual={int(test_targets.iloc[i])}, {'OK' if correct else 'WRONG'}")
    time.sleep(1)

with open(RESULTS_DIR / "financial_phase2_predictions.json", "w") as f:
    json.dump(predictions, f, indent=2)

# Calculate accuracy
correct_count = sum(1 for p in predictions.values() if p['prediction'] == p['actual'])
print(f"\nLLM Accuracy: {correct_count}/{n_test} = {correct_count/n_test:.1%}")


PHASE 2: PREDICTIONS
  Sample 0: pred=1, actual=0, WRONG
  Sample 1: pred=0, actual=1, WRONG
  Sample 2: pred=1, actual=1, OK
  Sample 3: pred=0, actual=1, WRONG
  Sample 4: pred=1, actual=1, OK

LLM Accuracy: 2/5 = 40.0%


## 6. Phase 3: Decision Rulebook

In [6]:
print("\n" + "="*60)
print("PHASE 3: RULEBOOK EXTRACTION")
print("="*60)

# Build comparison
comparison = ""
for i, p in predictions.items():
    comparison += f"Sample {i}: predicted={p['prediction']}, actual={p['actual']}\n"

prompt = f"""Extract decision rules from these predictions.

{comparison}

Give me 3-4 simple IF-THEN rules to predict market direction.
Format:
Rule 1: IF [condition] THEN [0 or 1]

Keep it simple and actionable."""

rulebook = call_llm(prompt, max_tokens=500)

if rulebook:
    with open(RESULTS_DIR / "financial_phase3_rulebook.txt", "w") as f:
        f.write(rulebook)
    print(f"\nPhase 3 complete! ({len(rulebook)} chars)")
    print(rulebook[:500])
else:
    print("Phase 3 failed!")


PHASE 3: RULEBOOK EXTRACTION

Phase 3 complete! (2039 chars)
1.  **Analyze the Request:**
    *   **Input:** A small dataset of 5 samples with `predicted` and `actual` values.
        *   Sample 0: pred=1, act=0 (Error)
        *   Sample 1: pred=0, act=1 (Error)
        *   Sample 2: pred=1, act=1 (Correct)
        *   Sample 3: pred=0, act=1 (Error)
        *   Sample 4: pred=1, act=1 (Correct)
    *   **Task:** Extract decision rules from these predictions.
    *   **Output Format:** 3-4 simple IF-THEN rules.
    *   **Goal:** Predict market direction 


## 7. Phase 4: Replicability Test

In [7]:
print("\n" + "="*60)
print("PHASE 4: REPLICABILITY TEST")
print("="*60)

if not rulebook:
    print("No rulebook. Skipping.")
else:
    replicability_results = {"samples": [], "domain": "financial"}
    
    for i in range(min(5, n_test)):
        row = test_features.iloc[i]
        
        prompt = f"""Apply these rules to predict market direction.
        
Rules:
{rulebook}
        
Features:
- return_1d: {row['return_1d']:.4f}
- rsi: {row['rsi']:.2f}
- volatility_5d: {row['volatility_5d']:.4f}
        
Output only the number 0 or 1."""
        
        resp = call_llm(prompt, max_tokens=10, temperature=0.0)
        
        pred = None
        if resp:
            if '1' in resp: pred = 1
            elif '0' in resp: pred = 0
        
        actual = int(test_targets.iloc[i])
        orig_pred = predictions[str(i)]['prediction']
        
        replicability_results["samples"].append({
            "index": i,
            "original": orig_pred,
            "fresh": pred,
            "actual": actual,
            "original_correct": orig_pred == actual if orig_pred is not None else None,
            "fresh_correct": pred == actual if pred is not None else None
        })
        
        print(f"  Sample {i}: orig={orig_pred}, fresh={pred}, actual={actual}")
        time.sleep(1)
    
    # Calculate scores
    n_samples = len(replicability_results["samples"])
    orig_correct = sum(1 for s in replicability_results["samples"] if s["original_correct"])
    fresh_correct = sum(1 for s in replicability_results["samples"] if s["fresh_correct"])
    
    replicability_results["original_accuracy"] = orig_correct / n_samples
    replicability_results["replicability_score"] = fresh_correct / n_samples
    replicability_results["n_samples"] = n_samples
    
    print(f"\nOriginal LLM Accuracy: {orig_correct}/{n_samples}")
    print(f"Fresh LLM Accuracy (Replicability Score): {fresh_correct}/{n_samples}")
    
    with open(RESULTS_DIR / "financial_replicability_test.json", "w") as f:
        json.dump(replicability_results, f, indent=2)


PHASE 4: REPLICABILITY TEST
  Sample 0: orig=1, fresh=1, actual=0
  Sample 1: orig=0, fresh=1, actual=1
  Sample 2: orig=1, fresh=1, actual=1
  Sample 3: orig=0, fresh=1, actual=1
  Sample 4: orig=1, fresh=1, actual=1

Original LLM Accuracy: 2/5
Fresh LLM Accuracy (Replicability Score): 4/5


## 8. Summary

In [8]:
print("\n" + "="*60)
print("LLMIP FINANCIAL PIPELINE COMPLETE")
print("="*60)

print("\n=== Comparison: Grid vs Financial ===")
print("Grid (stateless): XGBoost MAE = 0.0015 pu")
print("Financial (stateful): XGBoost Accuracy = 50%")
print("\nFinancial is MUCH harder - as expected!")

print("\n=== Output Files ===")
print(f"  Phase 1: {RESULTS_DIR / 'financial_phase1_analysis.txt'}")
print(f"  Phase 2: {RESULTS_DIR / 'financial_phase2_predictions.json'}")
print(f"  Phase 3: {RESULTS_DIR / 'financial_phase3_rulebook.txt'}")
print(f"  Phase 4: {RESULTS_DIR / 'financial_replicability_test.json'}")


LLMIP FINANCIAL PIPELINE COMPLETE

=== Comparison: Grid vs Financial ===
Grid (stateless): XGBoost MAE = 0.0015 pu
Financial (stateful): XGBoost Accuracy = 50%

Financial is MUCH harder - as expected!

=== Output Files ===
  Phase 1: /home/l1zle/LLMIP/results/financial_phase1_analysis.txt
  Phase 2: /home/l1zle/LLMIP/results/financial_phase2_predictions.json
  Phase 3: /home/l1zle/LLMIP/results/financial_phase3_rulebook.txt
  Phase 4: /home/l1zle/LLMIP/results/financial_replicability_test.json
